<a href="https://colab.research.google.com/github/razorbladesky/ruta-inundacion-ia/blob/main/simulacion_inundacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# ==========================================
# 1. CONSTANTES DE CONFIGURACIÓN
# ==========================================

# Representa un obstáculo infranqueable (pared o zona inaccesible)
BLOQUEADO = -1

# Niveles de inundación en las celdas transitables
SECO = 0
RIESGO_BAJO = 1
RIESGO_MEDIO = 2
RIESGO_ALTO = 3

# Diccionario de costos de entrada a cada tipo de celda transitable
# A mayor nivel de inundación, mayor es el esfuerzo (costo) para cruzar la celda
COSTOS = {
    SECO: 1,          # Tránsito normal
    RIESGO_BAJO: 3,   # Agua a los tobillos
    RIESGO_MEDIO: 6,  # Agua a las rodillas
    RIESGO_ALTO: 10   # Corriente fuerte, requiere mucho cuidado
}

# ==========================================
# 2. CREACIÓN DEL MAPA
# ==========================================

def crear_mapa_principal():
    """
    Crea y devuelve el mapa de la simulación de 10x10.
    Los valores representan el nivel de riesgo de inundación o si está bloqueado.
    """
    mapa_lista = [
        [ 0,  0,  0,  1,  0, -1,  0,  0,  1,  0],
        [ 0,  0,  1,  1,  0, -1,  0,  1,  0,  0],
        [ 0,  1,  1,  1,  1,  3,  1,  1,  1,  0],
        [ 0,  0,  2,  2,  1, -1,  1,  2,  0,  0],
        [ 0,  0,  2,  3,  1, -1,  1,  3,  2,  0],
        [ 0,  1,  2,  3,  2, -1,  2,  2,  1,  0],
        [ 0,  0,  1,  2,  1, -1,  1,  1,  0,  0],
        [ 0,  0,  0,  1,  0, -1,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  1,  1,  0, -1,  0,  1,  0,  0]
    ]
    return np.array(mapa_lista)

# ==========================================
# 3. COORDENADAS DE INICIO Y DESTINO
# ==========================================
# Formato (fila, columna)
INICIO = (2, 0)
REFUGIO = (2, 9)

# ==========================================
# 4. FUNCIÓN PARA IMPRIMIR EL MAPA
# ==========================================

def imprimir_mapa(mapa, inicio, refugio):
    """
    Imprime una representación visual del mapa en consola.
    - 'I' representa la posición inicial.
    - 'R' representa el refugio seguro.
    - '#' representa las zonas bloqueadas (-1).
    - Los números representan el nivel de inundación (0, 1, 2, 3).
    """
    filas, columnas = mapa.shape

    for f in range(filas):
        fila_texto = []
        for c in range(columnas):
            coordenada_actual = (f, c)
            valor = mapa[f, c]

            # Prioridad de dibujado: Inicio -> Refugio -> Obstáculos/Terreno
            if coordenada_actual == inicio:
                fila_texto.append('I')
            elif coordenada_actual == refugio:
                fila_texto.append('R')
            elif valor == BLOQUEADO:
                fila_texto.append('#')
            else:
                fila_texto.append(str(valor))

        # Unimos los elementos con un espacio para que se vea ordenado y cuadrado
        print(" ".join(fila_texto))

# ==========================================
# 5. EJECUCIÓN Y PRUEBA
# ==========================================

# Inicializamos el mapa
mapa_simulacion = crear_mapa_principal()

# Mostramos las dimensiones del mapa
print(f"Dimensiones del mapa generado: {mapa_simulacion.shape} (10 filas x 10 columnas)\n")

# Imprimimos la representación visual del mapa
print("Visualización del mapa (I = Inicio, R = Refugio, # = Bloqueado):")
imprimir_mapa(mapa_simulacion, INICIO, REFUGIO)

Dimensiones del mapa generado: (10, 10) (10 filas x 10 columnas)

Visualización del mapa (I = Inicio, R = Refugio, # = Bloqueado):
0 0 0 1 0 # 0 0 1 0
0 0 1 1 0 # 0 1 0 0
I 1 1 1 1 3 1 1 1 R
0 0 2 2 1 # 1 2 0 0
0 0 2 3 1 # 1 3 2 0
0 1 2 3 2 # 2 2 1 0
0 0 1 2 1 # 1 1 0 0
0 0 0 1 0 # 0 0 0 0
0 0 0 0 0 0 0 0 0 0
0 0 1 1 0 # 0 1 0 0


In [2]:
def obtener_vecinos(mapa, celda):
    """
    Devuelve los vecinos válidos de una celda en el siguiente orden fijo:
    arriba, derecha, abajo, izquierda.
    Un vecino es válido si está dentro de los límites y no está BLOQUEADO.
    """
    fila, col = celda
    filas_max, columnas_max = mapa.shape
    vecinos_validos = []

    # Movimientos posibles en el orden requerido:
    # Arriba (-1, 0), Derecha (0, +1), Abajo (+1, 0), Izquierda (0, -1)
    movimientos = [
        (fila - 1, col),  # Arriba
        (fila, col + 1),  # Derecha
        (fila + 1, col),  # Abajo
        (fila, col - 1)   # Izquierda
    ]

    for f, c in movimientos:
        # 1. Verificar límites de la matriz
        if 0 <= f < filas_max and 0 <= c < columnas_max:
            # 2. Verificar que no esté bloqueado
            if mapa[f, c] != BLOQUEADO:
                vecinos_validos.append((f, c))

    return vecinos_validos


def costo_de_ruta(mapa, ruta):
    """
    Calcula el costo total de transitar por una ruta.
    La primera celda (inicio) no suma costo.
    Retorna None si la ruta es inválida, vacía o toca zonas bloqueadas.
    """
    # Validación de ruta nula o vacía
    if ruta is None or len(ruta) < 1:
        return None

    costo_total = 0

    # Recorremos cada celda de la ruta indicando su posición (índice)
    for i, celda in enumerate(ruta):
        f, c = celda
        valor_celda = mapa[f, c]

        # Si la celda está bloqueada, la ruta es inválida
        if valor_celda == BLOQUEADO:
            return None

        # El inicio (índice 0) no suma costo
        if i > 0:
            costo_total += COSTOS[valor_celda]

    return costo_total

# ==========================================
# PRUEBAS UNITARIAS (ASSERTS)
# ==========================================

# Usamos el mapa cargado en la celda anterior
mapa = mapa_simulacion

# Prueba 1: Vecinos en (2, 0) -> Arriba, Derecha, Abajo (Izquierda está fuera de límites)
assert obtener_vecinos(mapa, (2, 0)) == [(1, 0), (2, 1), (3, 0)], "Error en vecinos de (2, 0)"

# Prueba 2: Vecinos en (0, 0) -> Derecha, Abajo (Arriba e Izquierda fuera de límites)
assert obtener_vecinos(mapa, (0, 0)) == [(0, 1), (1, 0)], "Error en vecinos de (0, 0)"

# Prueba 3: Vecinos en (1, 4) -> Arriba, Abajo, Izquierda (Derecha (1, 5) es bloqueado '#')
assert obtener_vecinos(mapa, (1, 4)) == [(0, 4), (2, 4), (1, 3)], "Error en vecinos de (1, 4)"

# Prueba 4: Costo de la ruta directa horizontal de fila 2: desde (2, 0) hasta (2, 9)
ruta_fila_2 = [(2, i) for i in range(10)]
assert costo_de_ruta(mapa, ruta_fila_2) == 32, f"Error: se esperaba costo 32, pero se obtuvo {costo_de_ruta(mapa, ruta_fila_2)}"

# Prueba 5: Costo de ruta con una sola celda (solo inicio)
assert costo_de_ruta(mapa, [(2, 0)]) == 0, "Error: la ruta de un solo paso debe costar 0"

# Prueba 6: Ruta que pasa por un obstáculo bloqueado (0, 5)
ruta_con_bloqueo = [(0, 4), (0, 5), (0, 6)]
assert costo_de_ruta(mapa, ruta_con_bloqueo) is None, "Error: ruta con bloqueos debe retornar None"

print("Parte 2: todas las pruebas pasaron")

Parte 2: todas las pruebas pasaron
